<a href="https://colab.research.google.com/github/25ds3000042/LangChain-Chains-Retrieval-and-RAG/blob/main/TechNova_RAG_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
%pip install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [3]:
from google.colab import files

uploaded = files.upload()

Saving technova_policy.txt to technova_policy (1).txt


In [4]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Groq API key loaded successfully.")

Groq API key loaded successfully.


In [5]:

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import os


/tmp/ipykernel_450/3096823361.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [6]:
# 1. Load the policy document
documents = TextLoader(
    "/content/technova_policy.txt",
    encoding="utf-8"
).load()

print("Document loaded:", len(documents), "document(s)")

Document loaded: 1 document(s)


In [7]:
# 2. Split the document into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = splitter.split_documents(documents)
print("Number of chunks:", len(chunks))

Number of chunks: 2


In [8]:
# 3. Generate embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
# 4. Store embeddings in the FAISS vector database
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created successfully.")

FAISS vector store created successfully.


In [10]:
# 5. Create a retriever that returns the top 3 chunks
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [11]:
# 6. Configure the Groq language model
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [12]:
# 7. Create the RAG prompt template
prompt = ChatPromptTemplate.from_template("""
You answer questions using only the TechNova Solutions
employee policy context provided below.

Do not invent policies or use outside knowledge.
If the answer is absent from the context, say:
"The provided document does not specify this information."

Context:
{context}

Question:
{question}

Answer:
""")

In [13]:
# Combine retrieved chunks into context
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [14]:
# 8. Build the RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
# 9. Test the three assignment questions
questions = [
    "How many paid annual leave days does a full-time employee receive per year?",
    "An employee wants remote work on Wednesday and leave on Friday. What do policies say?",
    "What is the parental leave policy?"
]

for i, question in enumerate(questions, start=1):
    answer = rag_chain.invoke(question)

    print(f"\nQuestion {i}: {question}")
    print("Answer:", answer)

# 10. Expected answers from the supplied policy document
#
# Q1: A full-time employee receives 18 paid annual leave
# days per calendar year.
#
# Q2: Remote work is allowed on Wednesdays and Fridays,
# subject to manager approval and an email at least one
# working day in advance. Annual leave must be requested
# at least 3 calendar days before the leave date and
# depends on manager confirmation.
#
# Q3: The provided document does not specify this information.



Question 1: How many paid annual leave days does a full-time employee receive per year?
Answer: A full‑time employee receives **18 paid annual leave days per calendar year**.

Question 2: An employee wants remote work on Wednesday and leave on Friday. What do policies say?
Answer: **Remote work on Wednesday**  
- The policy permits employees to work from home on Wednesdays.  
- The employee must **email the manager at least one working day in advance** to request remote work.  
- Remote work is **subject to manager approval**.

**Leave on Friday**  
- The policy allows full‑time employees to take paid annual leave.  
- The employee must **submit the leave request to the manager at least 3 calendar days before the requested leave date** (i.e., before the Friday).  
- The leave is granted only after the **manager’s confirmation**.

**Combined situation**  
- The employee can request remote work for Wednesday following the email‑in‑advance and approval steps.  
- For Friday, the employee